# Ammu Cross-Encoder Family Evaluation on Off On Topic Guardrail Dataset

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                     Off On Topic Guardrail Benchmark Dataset                           │
│               (Hugging Face: kalyan-ks/off-on-topic-guardrail, test split)             │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│             Fine-Tuned Ammu Cross-Encoder Family Evaluation (Colab GPU)                │
│                                                                                        │
│   • Ammu-1.0-base  : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-base (~149M params)       │
│   • Ammu-1.0-small : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-small (~68M params)        │
│   • Ammu-1.0-tiny  : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-tiny  (~28M params)        │
│   • Ammu-1.0-nano  : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-nano  (~14M params)        │
│                                                                                        │
│   - Hugging Face Pipeline: High-level text-classification pipeline for clean inference │
│   - Native Labels as IDs: 0 = Off Topic, 1 = On Topic (no redundant string mapping)   │
│   - Paired-Input Cross-Encoder: Native {'text': system_prompt, 'text_pair': user_prompt}│
│   - Truncation Safety: Longest-first truncation preserves user query on long prompts   │
│   - Colab GPU Acceleration: FP32 full precision (matching training) with VRAM garbage collection│
│   - Stream-to-Disk Checkpointing: Real-time response caching for instant resume        │
└───────────────────────────────────────────┬────────────────────────────────────────────┘
                                            │
                                            ▼
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                          Evaluation Metrics Computation                                │
│             • Difficulty Tier-wise Macro-F1 (Easy, Medium, Hard)                       │
│             • Overall Macro-F1                                                         │
│             • Comparative Leaderboard Exported to CSV                                  │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

## 1. Install Dependencies (VS Code / Colab GPU Runtime)

In [1]:
# ==============================================================================
# 1. INSTALL DEPENDENCIES (FOR COLAB GPU RUNTIME)
# ==============================================================================

#!pip install uv
#!uv pip install --system  torch transformers accelerate datasets scikit-learn pandas

## 2. Imports and Environment Configuration

In [2]:
# ==============================================================================
# 2. IMPORTS AND ENVIRONMENT CONFIGURATION
# ==============================================================================
import os
# Suppress Hugging Face Windows symlink warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import gc
import json
import time
from pathlib import Path
from typing import Dict, List, Any, Optional

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from datasets import load_dataset

import torch
from transformers import pipeline

# Load environment variables from .env if present
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN", "")

# Colab GPU Hardware Acceleration Detection
print("=" * 75)
print("EXECUTION ENVIRONMENT & COLAB GPU ACCELERATION")
print("=" * 75)
HAS_CUDA = torch.cuda.is_available()
DEVICE_SPEC = 0 if HAS_CUDA else -1
TORCH_DTYPE = torch.float32  # Strictly FP32 precision (matching training)

print(f" - CUDA Available   : {HAS_CUDA}")
if HAS_CUDA:
    print(f" - Colab GPU Model  : {torch.cuda.get_device_name(0)}")
    print(f" - Total VRAM       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f" - Inference Dtype  : {TORCH_DTYPE}")
else:
    print(" - Device Note      : Running on CPU. Please switch Colab runtime to GPU (T4 / L4 / A100).")
if HF_TOKEN:
    masked_token = HF_TOKEN[:7] + "..." + HF_TOKEN[-4:] if len(HF_TOKEN) > 11 else "[SET]"
    print(f" - HF_TOKEN         : {masked_token} (Higher rate limits enabled)")
else:
    print(" - HF_TOKEN         : [NOT SET] (Public models will download fine)")
print("=" * 75)

# Checkpoint and output directory
RESULTS_DIR = Path("evaluation_results_encoders")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

EXECUTION ENVIRONMENT & COLAB GPU ACCELERATION
 - CUDA Available   : True
 - Colab GPU Model  : NVIDIA L4
 - Total VRAM       : 22.03 GB
 - Inference Dtype  : torch.float32
 - HF_TOKEN         : [NOT SET] (Public models will download fine)


## 3. Load Benchmark Dataset Directly from Hugging Face Hub

We load the test split of `kalyan-ks/off-on-topic-guardrail` directly from Hugging Face Hub via `data_files={"test": "test.csv"}`.
Labels are numeric IDs (`0` for Off Topic, `1` for On Topic).

In [3]:
# ==============================================================================
# 3. LOAD BENCHMARK DATASET (TEST SPLIT DIRECTLY FROM HUGGING FACE HUB)
# ==============================================================================
DATASET_REPO = "kalyan-ks/off-on-topic-guardrail"
print(f"Loading '{DATASET_REPO}' test split directly from Hugging Face Hub...")

hf_auth_token = HF_TOKEN if HF_TOKEN else None
raw_test_dataset = load_dataset(
    DATASET_REPO,
    data_files={"test": "test.csv"},
    split="test",
    token=hf_auth_token,
)

eval_benchmark_df = raw_test_dataset.to_pandas()
print(f"Successfully loaded test split: {len(eval_benchmark_df):,} instances.")

# Ensure instance_id exists
if "instance_id" not in eval_benchmark_df.columns:
    eval_benchmark_df["instance_id"] = eval_benchmark_df.index.astype(str)
else:
    eval_benchmark_df["instance_id"] = eval_benchmark_df["instance_id"].astype(str)

# Ensure label is integer ID: 0 (Off Topic) and 1 (On Topic)
if eval_benchmark_df["label"].dtype == object:
    eval_benchmark_df["label"] = eval_benchmark_df["label"].map({"Off Topic": 0, "On Topic": 1, "On-Topic": 1, "0": 0, "1": 1}).fillna(eval_benchmark_df["label"])
eval_benchmark_df["label"] = eval_benchmark_df["label"].astype(int)

# Standardize difficulty tier
# Map difficulty_tier (0 -> Easy, 1 -> Medium, 2 -> Hard) to standardized tier names
TIER_MAP = {0: "Easy", 1: "Medium", 2: "Hard", "0": "Easy", "1": "Medium", "2": "Hard", "easy": "Easy", "medium": "Medium", "hard": "Hard", "Easy": "Easy", "Medium": "Medium", "Hard": "Hard"}
eval_benchmark_df["difficulty_tier"] = eval_benchmark_df["difficulty_tier"].map(TIER_MAP).fillna(eval_benchmark_df["difficulty_tier"])

# Display dataset summary
tier_summary = eval_benchmark_df.groupby(["difficulty_tier", "label"]).size().unstack(fill_value=0)
tier_summary.columns = ["Off Topic (0)", "On Topic (1)"]
tier_summary["Total"] = tier_summary.sum(axis=1)
print("\nTest Split Breakdown by Difficulty Tier and Label:")
print(tier_summary)
print(f"\nTotal Evaluation Instances: {len(eval_benchmark_df):,}")

Loading 'kalyan-ks/off-on-topic-guardrail' test split directly from Hugging Face Hub...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Successfully loaded test split: 4,008 instances.

Test Split Breakdown by Difficulty Tier and Label:
                 Off Topic (0)  On Topic (1)  Total
difficulty_tier                                    
Easy                       299           336    635
Hard                       852           950   1802
Medium                     734           837   1571

Total Evaluation Instances: 4,008


## 4. Ammu Encoder Models Registry

We evaluate all four fine-tuned cross-encoder models:
1. **Ammu-1.0-base** (`kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-base`)
2. **Ammu-1.0-small** (`kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-small`)
3. **Ammu-1.0-tiny** (`kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-tiny`)
4. **Ammu-1.0-nano** (`kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-nano`)

Optimized batch sizes are configured for Colab GPU (T4 / L4 / A100).

In [4]:
# ==============================================================================
# 4. ENCODER MODEL REGISTRY CONFIGURATION
# ==============================================================================

MAX_SEQUENCE_LENGTH = 1024  # Max token budget (longest_first preserves user prompt)

MODEL_REGISTRY: Dict[str, Dict[str, Any]] = {
    "Ammu-1.0-base": {
        "hf_model_id": "kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-base",
        "file_slug": "ammu_1_0_base",
        "description": "Ammu 1.0 Base Cross-Encoder (~149M params, ModernBert architecture)",
        "batch_size": 64,
    },
    "Ammu-1.0-small": {
        "hf_model_id": "kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-small",
        "file_slug": "ammu_1_0_small",
        "description": "Ammu 1.0 Small Cross-Encoder (~68M params, ModernBert architecture)",
        "batch_size": 64,
    },
    "Ammu-1.0-tiny": {
        "hf_model_id": "kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-tiny",
        "file_slug": "ammu_1_0_tiny",
        "description": "Ammu 1.0 Tiny Cross-Encoder (~28M params, ModernBert architecture)",
        "batch_size": 64,
    },
    "Ammu-1.0-nano": {
        "hf_model_id": "kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-nano",
        "file_slug": "ammu_1_0_nano",
        "description": "Ammu 1.0 Nano Cross-Encoder (~14M params, ModernBert architecture)",
        "batch_size": 64,
    },
}

# Aliases for easy model selection
MODEL_ALIASES: Dict[str, str] = {
    "base": "Ammu-1.0-base",
    "ammu-base": "Ammu-1.0-base",
    "small": "Ammu-1.0-small",
    "ammu-small": "Ammu-1.0-small",
    "tiny": "Ammu-1.0-tiny",
    "ammu-tiny": "Ammu-1.0-tiny",
    "nano": "Ammu-1.0-nano",
    "ammu-nano": "Ammu-1.0-nano",
}

print("Configured Model Registry:")
for name, cfg in MODEL_REGISTRY.items():
    print(f"  • {name:<15} : {cfg['hf_model_id']} (Colab GPU Batch Size: {cfg['batch_size']})")

Configured Model Registry:
  • Ammu-1.0-base   : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-base (Colab GPU Batch Size: 64)
  • Ammu-1.0-small  : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-small (Colab GPU Batch Size: 64)
  • Ammu-1.0-tiny   : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-tiny (Colab GPU Batch Size: 64)
  • Ammu-1.0-nano   : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-nano (Colab GPU Batch Size: 64)


## 5. Batch Inference Engine using Hugging Face `pipeline`

Utilizing Hugging Face's high-level `pipeline("text-classification")`for inference:
- **Native Paired Input**: Passed as `{"text": system_prompt, "text_pair": user_prompt}` directly to the pipeline.
- **Longest-First Truncation**: Automatically trims the longer system prompt first, guaranteeing that the user prompt is never truncated.
- **Colab GPU Batching**: Utilizes `device=0` with `torch.float32 (FP32)` and batching for high inference speeds on Colab GPUs.
- **Labels as IDs**: Predictions are output as integer IDs (`0` = Off Topic, `1` = On Topic).
- **VRAM Management**: Cleans up model memory and calls `torch.cuda.empty_cache()` between runs to prevent GPU out-of-memory errors.
- **Stream-to-Disk Checkpointing**: Predictions are cached in `evaluation_results_encoders/off_on_topic_guardrail_responses_{file_slug}.csv` for instant resume.

In [5]:
# ==============================================================================
# 5. PIPELINE-BASED BATCH INFERENCE ENGINE (WITH CHECKPOINTING & RESUME)
# ==============================================================================

def load_existing_encoder_responses(responses_file: Path) -> Dict[str, Dict[str, Any]]:
    """Loads existing cached completions from CSV checkpoint file for resume capability."""
    existing = {}
    if responses_file.exists():
        try:
            cached_df = pd.read_csv(responses_file)
            for _, row in cached_df.iterrows():
                inst_id = str(row.get("instance_id", ""))
                if inst_id and row.get("status") == "success":
                    existing[inst_id] = row.to_dict()
        except Exception as e:
            print(f"Warning reading checkpoint file {responses_file}: {e}")
    return existing


def run_encoder_model_inference(canonical_name: str, eval_df: pd.DataFrame) -> pd.DataFrame:
    """Runs streamlined batch inference using Hugging Face pipeline with Colab GPU acceleration."""
    cfg = MODEL_REGISTRY[canonical_name]
    hf_model_id = cfg["hf_model_id"]
    file_slug = cfg["file_slug"]
    batch_size = cfg["batch_size"]
    
    responses_file = RESULTS_DIR / f"off_on_topic_guardrail_responses_{file_slug}.csv"
    existing_responses = load_existing_encoder_responses(responses_file)
    print(f"[{canonical_name}] Found {len(existing_responses):,} existing completions in checkpoint.")
    
    pending_df = eval_df[~eval_df["instance_id"].astype(str).isin(existing_responses)].copy()
    print(f"[{canonical_name}] Instances pending inference: {len(pending_df):,} / {len(eval_df):,}")
    
    if len(pending_df) > 0:
        print(f"Initializing HF text-classification pipeline: {hf_model_id}...")
        
        # Initialize Hugging Face pipeline with Colab GPU device and FP32
        classifier = pipeline(
            "text-classification",
            model=hf_model_id,
            device=DEVICE_SPEC,
            torch_dtype=TORCH_DTYPE,
            token=hf_auth_token,
            truncation="longest_first",
            max_length=MAX_SEQUENCE_LENGTH,
        )
        
        # Format paired inputs: {'text': system_prompt, 'text_pair': user_prompt}
        input_pairs = [
            {"text": sys_prompt, "text_pair": user_prompt}
            for sys_prompt, user_prompt in zip(pending_df["system_prompt"], pending_df["user_prompt"])
        ]
        
        print(f"Running batch inference (Batch Size: {batch_size}, Device: {DEVICE_SPEC}, Dtype: {TORCH_DTYPE})...")
        
        # Run inference using pipeline with tqdm progress bar
        pipeline_results = []
        for out in tqdm(
            classifier(input_pairs, batch_size=batch_size, truncation=True, max_length=MAX_SEQUENCE_LENGTH),
            total=len(input_pairs),
            desc=f"Inference: {canonical_name}",
        ):
            pipeline_results.append(out)
            
        # Parse results and update responses dictionary
        for (_, row), out in zip(pending_df.iterrows(), pipeline_results):
            raw_label = str(out.get("label", ""))
            # 0 for Off Topic, 1 for On Topic
            pred_id = 1 if ("1" in raw_label or "on" in raw_label.lower()) else 0
                
            inst_id = str(row["instance_id"])
            record = {
                "instance_id": inst_id,
                "difficulty_tier": str(row["difficulty_tier"]),
                "label": int(row["label"]),
                "prediction": pred_id,
                "model": canonical_name,
                "status": "success",
            }
            existing_responses[inst_id] = record
            
        # Release classifier and clean GPU VRAM on Colab
        del classifier
        if HAS_CUDA:
            torch.cuda.empty_cache()
        gc.collect()
        
        # Save / update checkpoint file
        all_df = pd.DataFrame(list(existing_responses.values()))
        all_df.to_csv(responses_file, index=False, encoding="utf-8")
        print(f"Inference complete for {canonical_name}! Checkpoint updated: {responses_file}")
        
    # Load and format full results dataframe
    res_df = pd.DataFrame(list(existing_responses.values()))
    tier_norm_map = {0: "Easy", 1: "Medium", 2: "Hard", "0": "Easy", "1": "Medium", "2": "Hard", "easy": "Easy", "medium": "Medium", "hard": "Hard", "Easy": "Easy", "Medium": "Medium", "Hard": "Hard"}
    res_df["difficulty_tier"] = res_df["difficulty_tier"].map(tier_norm_map).fillna(res_df["difficulty_tier"])
    res_df["instance_id"] = res_df["instance_id"].astype(str)
    res_df["label"] = res_df["label"].astype(int)
    res_df["prediction"] = res_df["prediction"].astype(int)
    return res_df

## 6. Metric Computation Engine (Difficulty Tier-wise & Overall Macro-F1)

We evaluate model performance strictly using **Macro-F1** across label IDs `[0, 1]`:
- **Easy Macro-F1 (%)**
- **Medium Macro-F1 (%)**
- **Hard Macro-F1 (%)**
- **Overall Macro-F1 (%)**

*(Accuracy and Throughput are intentionally omitted as per benchmark specifications.)*

In [6]:
# ==============================================================================
# 6. DIFFICULTY TIER-WISE AND OVERALL MACRO-F1 CALCULATOR
# ==============================================================================

def compute_encoder_metrics(results_df: pd.DataFrame, model_name: str) -> Dict[str, Any]:
    """Calculates difficulty tier-wise and overall Macro-F1 scores across label IDs [0, 1]."""
    metrics = {
        "Model": model_name,
    }
    
    # Standardize difficulty_tier to 'Easy', 'Medium', 'Hard' regardless of whether encoded as 0/1/2 or text
    tier_map = {
        0: "Easy", 1: "Medium", 2: "Hard",
        "0": "Easy", "1": "Medium", "2": "Hard",
        "easy": "Easy", "medium": "Medium", "hard": "Hard",
        "Easy": "Easy", "Medium": "Medium", "Hard": "Hard",
    }
    normalized_tiers = (
        results_df["difficulty_tier"]
        .map(tier_map)
        .fillna(results_df["difficulty_tier"])
        .astype(str)
        .str.strip()
        .str.capitalize()
    )
    
    # Difficulty tier-wise Macro-F1
    for tier in ["Easy", "Medium", "Hard"]:
        tier_subset = results_df[normalized_tiers == tier]
        if len(tier_subset) > 0:
            f1 = f1_score(
                tier_subset["label"],
                tier_subset["prediction"],
                average="macro",
                labels=[0, 1],
                zero_division=0,
            )
            metrics[f"{tier} Macro-F1 (%)"] = round(f1 * 100, 2)
        else:
            metrics[f"{tier} Macro-F1 (%)"] = np.nan
            
    # Overall Macro-F1
    overall_f1 = f1_score(
        results_df["label"],
        results_df["prediction"],
        average="macro",
        labels=[0, 1],
        zero_division=0,
    )
    metrics["Overall Macro-F1 (%)"] = round(overall_f1 * 100, 2)
    
    return metrics


## 7. Model Execution & Evaluation Loop

Choose your target model to evaluate individually, or set `SELECTED_MODEL = "All"` to benchmark all four Ammu cross-encoders sequentially.

*Available Options:*
- `"Ammu-1.0-base"`
- `"Ammu-1.0-small"`
- `"Ammu-1.0-tiny"`
- `"Ammu-1.0-nano"`
- `"All"` (Evaluates all 4 models)

In [7]:
# ==============================================================================
# 7. SELECT MODEL AND EXECUTE EVALUATION
# ==============================================================================
SELECTED_MODEL = "All"   # Options: "Ammu-1.0-base" | "Ammu-1.0-small" | "Ammu-1.0-tiny" | "Ammu-1.0-nano" | "All"

# Determine target models to evaluate
if SELECTED_MODEL.strip().lower() == "all":
    models_to_run = list(MODEL_REGISTRY.keys())
else:
    raw_key = SELECTED_MODEL.strip().lower()
    canonical_name = MODEL_ALIASES.get(raw_key)
    if canonical_name is None:
        for k in MODEL_REGISTRY:
            if k.lower() == raw_key:
                canonical_name = k
                break
    if canonical_name is None:
        raise ValueError(f"Unknown model '{SELECTED_MODEL}'. Choose from: {list(MODEL_REGISTRY.keys()) + ['All']}")
    models_to_run = [canonical_name]

print(f"Models scheduled for evaluation ({len(models_to_run)}): {models_to_run}")

all_encoder_metrics = []

# Execute evaluation loop
for target_model in models_to_run:
    print("\n" + "=" * 75)
    print(f"STARTING BENCHMARK: {target_model}")
    print(f"HF Model ID       : {MODEL_REGISTRY[target_model]['hf_model_id']}")
    print("=" * 75)
    
    # Run batch inference with checkpointing
    preds_df = run_encoder_model_inference(target_model, eval_benchmark_df)
    
    # Calculate Macro-F1
    metric_dict = compute_encoder_metrics(preds_df, target_model)
    all_encoder_metrics.append(metric_dict)
    
    print(f"\nMetrics for {target_model}:")
    for k, v in metric_dict.items():
        print(f"  • {k}: {v}")

Models scheduled for evaluation (4): ['Ammu-1.0-base', 'Ammu-1.0-small', 'Ammu-1.0-tiny', 'Ammu-1.0-nano']

STARTING BENCHMARK: Ammu-1.0-base
HF Model ID       : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-base
[Ammu-1.0-base] Found 4,008 existing completions in checkpoint.
[Ammu-1.0-base] Instances pending inference: 0 / 4,008

Metrics for Ammu-1.0-base:
  • Model: Ammu-1.0-base
  • Easy Macro-F1 (%): 97.15
  • Medium Macro-F1 (%): 90.71
  • Hard Macro-F1 (%): 90.3
  • Overall Macro-F1 (%): 91.56

STARTING BENCHMARK: Ammu-1.0-small
HF Model ID       : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-small
[Ammu-1.0-small] Found 4,008 existing completions in checkpoint.
[Ammu-1.0-small] Instances pending inference: 0 / 4,008

Metrics for Ammu-1.0-small:
  • Model: Ammu-1.0-small
  • Easy Macro-F1 (%): 96.37
  • Medium Macro-F1 (%): 88.93
  • Hard Macro-F1 (%): 89.44
  • Overall Macro-F1 (%): 90.35

STARTING BENCHMARK: Ammu-1.0-tiny
HF Model ID       : kalyan-ks/Ammu-1.0-OffOnTopic-Guardrail-tiny
[A

## 8. Summary Leaderboard & Report Export

We aggregate the tier-wise and overall Macro-F1 results across all evaluated encoder models, sort them descending by `Overall Macro-F1 (%)`, display the styled leaderboard table, and export to CSV.

In [8]:
# ==============================================================================
# 8. SUMMARY LEADERBOARD AND EXPORT
# ==============================================================================

if len(all_encoder_metrics) > 0:
    leaderboard_df = pd.DataFrame(all_encoder_metrics)
    
    # Sort descending by Overall Macro-F1
    leaderboard_df.sort_values(by="Overall Macro-F1 (%)", ascending=False, inplace=True)
    leaderboard_df.reset_index(drop=True, inplace=True)
    leaderboard_df.index = leaderboard_df.index + 1  # 1-based rank
    
    # Save to CSV
    export_csv_path = RESULTS_DIR / "encoder_evaluation_leaderboard.csv"
    leaderboard_df.to_csv(export_csv_path, index=True, index_label="Rank", encoding="utf-8")
    print(f"\nLeaderboard exported successfully to: {export_csv_path.resolve()}")
    
    # Display formatted leaderboard
    print("\n" + "=" * 85)
    print("          AMMU ENCODER GUARDRAIL - EVALUATION LEADERBOARD (Macro-F1 %)")
    print("=" * 85)
    print(leaderboard_df.to_string())
    print("=" * 85)
    
    # Render styled HTML table in notebook with green highlight for top performer
    styled_table = (
        leaderboard_df.style
        .highlight_max(subset=["Overall Macro-F1 (%)", "Easy Macro-F1 (%)", "Medium Macro-F1 (%)", "Hard Macro-F1 (%)"], color="#d4edda")
        .format({
            "Easy Macro-F1 (%)": "{:.2f}",
            "Medium Macro-F1 (%)": "{:.2f}",
            "Hard Macro-F1 (%)": "{:.2f}",
            "Overall Macro-F1 (%)": "{:.2f}",
        })
        .set_caption("Off On Topic Guardrail - Ammu Cross-Encoder Benchmark Leaderboard")
    )
    display(styled_table)
else:
    print("No models were evaluated.")


Leaderboard exported successfully to: /content/evaluation_results_encoders/encoder_evaluation_leaderboard.csv

          AMMU ENCODER GUARDRAIL - EVALUATION LEADERBOARD (Macro-F1 %)
            Model  Easy Macro-F1 (%)  Medium Macro-F1 (%)  Hard Macro-F1 (%)  Overall Macro-F1 (%)
1   Ammu-1.0-base              97.15                90.71              90.30                 91.56
2   Ammu-1.0-tiny              95.74                89.66              90.19                 90.87
3  Ammu-1.0-small              96.37                88.93              89.44                 90.35
4   Ammu-1.0-nano              91.77                86.55              86.39                 87.31


,Model,Easy Macro-F1 (%),Medium Macro-F1 (%),Hard Macro-F1 (%),Overall Macro-F1 (%)
1,Ammu-1.0-base,97.15,90.71,90.30,91.56
2,Ammu-1.0-tiny,95.74,89.66,90.19,90.87
3,Ammu-1.0-small,96.37,88.93,89.44,90.35
4,Ammu-1.0-nano,91.77,86.55,86.39,87.31
